In [24]:
# %pip install python-dotenv
# %uv add dspy

In [25]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


### check aicodetools library

In [26]:
import time
import threading
import tiktoken
from collections import deque
import dspy
from dspy.utils.callback import BaseCallback


class SlidingWindowLimiter:
    """Rate limiter that enforces both request and token limits per rolling minute."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, max_requests_per_min=1000, max_tokens_per_min=2_000_000):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.max_requests = max_requests_per_min
            cls._instance.max_tokens = max_tokens_per_min
            cls._instance.requests = deque()  # [(timestamp, tokens)]
            cls._instance._lock = threading.Lock()
            cls._instance.encoder = tiktoken.get_encoding("cl100k_base")
        return cls._instance

    def _cleanup(self, now):
        """Remove entries older than 60s."""
        while self.requests and now - self.requests[0][0] > 60:
            self.requests.popleft()

    def _count(self):
        """Total requests & tokens in current 60s window."""
        total_tokens = sum(t for _, t in self.requests)
        return len(self.requests), total_tokens

    def acquire(self, tokens_used=0):
        """Wait until request fits in sliding 60s window."""
        with self._lock:
            while True:
                now = time.time()
                self._cleanup(now)
                req_count, token_count = self._count()

                # Can fit in current 60s window?
                if (req_count < self.max_requests and
                        token_count + tokens_used <= self.max_tokens):
                    # Record the new request
                    self.requests.append((now, tokens_used))
                    break  # proceed

                # Otherwise, figure out when we can retry
                oldest_time = self.requests[0][0]
                sleep_time = max(0.01, 60 - (now - oldest_time))
                print(f"⚠️ Throttling: sleeping {sleep_time:.2f}s (req={req_count}, tokens={token_count})")
                time.sleep(sleep_time)


class DelayAndLogCallback(BaseCallback):
    """DSPy callback using sliding window limiter."""

    def __init__(self):
        self.limiter = SlidingWindowLimiter()

    def _estimate_tokens(self, messages=None, prompt=None):
        """Estimate token usage using tiktoken."""
        text = ""
        if prompt:
            text = str(prompt)
        elif messages:
            # concatenate all message contents
            text = " ".join(m.get("content", "") for m in messages)
        return len(self.limiter.encoder.encode(text))

    def on_lm_start(self, *args, **kwargs):
        inputs = kwargs.get("inputs") or {}
        prompt = inputs.get("prompt")
        messages = inputs.get("messages")
        tokens_used = self._estimate_tokens(messages=messages, prompt=prompt)
        self.limiter.acquire(tokens_used=tokens_used)

    def on_lm_end(self, *args, **kwargs):
        pass


In [27]:

# from aicodetools import ClientManager 

# code_tool_manager = ClientManager(
#                 "super-bench:latest", base_log_dir="runs/super/"
#             )

# code_tool_client = code_tool_manager.get_client('initial')

In [28]:
import os
# os.environ['OPENAI_API_KEY'/] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [29]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=30, callbacks=[DelayAndLogCallback()])
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [30]:
# print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "t : Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [31]:
from gepa_artifact.benchmarks.researchcodebench import benchmark as rc_metas

In [32]:
bench = rc_metas[0].benchmark()

KeyboardInterrupt: 

In [ ]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(34, 34, 144)

In [ ]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'paper_id': 'DiffusionDPO', 'problem_root_rel': 'pset/DiffusionDPO', 'annotated_file_path': 'loss.py', 'snippet_name': 'calculate model losses', 'start_line': 33, 'end_line': 39, 'masked_file': 'import torch\nimport torch.nn.functional as F\n\ndef compute_loss(model_pred, target, args, ref_unet=None, model_batch_args=None, added_cond_kwargs=None):\n    """\n    Compute the loss for either SFT or DPO training.\n    \n    Args:\n        model_pred: The prediction from the model being trained\n        target: The target (typically noise) the model is trying to predict\n        args: The arguments containing training configuration\n        ref_unet: The reference UNet model (required for DPO)\n        model_batch_args: Arguments to pass to the UNet models (required for DPO)\n        added_cond_kwargs: Additional conditioning kwargs (required for DPO with SDXL)\n    \n    Returns:\n        loss: The computed loss\n        metrics: Dictionary of additional metrics for logging\n     

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [ ]:
program = rc_metas[0].program[0]
program

model = Predict(RCBResponse(instance_id, paper_id, snippet_name, masked_file, context_files, paper -> result
    instructions='Solve the question and provide the answer in the correct format.'
    instance_id = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Instance Id:', 'desc': '${instance_id}'})
    paper_id = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Paper Id:', 'desc': '${paper_id}'})
    snippet_name = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Snippet Name:', 'desc': '${snippet_name}'})
    masked_file = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Masked File:', 'desc': '${masked_file}'})
    context_files = Field(annotation=list required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Context Files:', 'desc': '${context_files}'})
    paper = Field(annotation=st

### Make Sure docker is installed and running

## Define an evaluator and evaluate the base program

In [ ]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=rc_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json=''
)

In [ ]:
base_score = evaluate(program)

Average Metric: 2.00 / 26 (7.7%):  18%|█▊        | 26/144 [01:01<10:00,  5.09s/it]⚠️ Throttling: sleeping 0.02s (req=43, tokens=1981480)
⚠️ Throttling: sleeping 0.16s (req=43, tokens=1999822)
⚠️ Throttling: sleeping 0.02s (req=42, tokens=1969848)
⚠️ Throttling: sleeping 0.21s (req=42, tokens=1988116)
⚠️ Throttling: sleeping 0.03s (req=41, tokens=1958573)
⚠️ Throttling: sleeping 0.05s (req=41, tokens=1977242)
⚠️ Throttling: sleeping 0.04s (req=41, tokens=1977079)
⚠️ Throttling: sleeping 3.79s (req=41, tokens=1977092)
Average Metric: 2.00 / 29 (6.9%):  20%|██        | 29/144 [01:06<05:11,  2.71s/it]⚠️ Throttling: sleeping 0.05s (req=41, tokens=1977057)
⚠️ Throttling: sleeping 0.05s (req=41, tokens=1976831)
⚠️ Throttling: sleeping 0.04s (req=41, tokens=1976766)
⚠️ Throttling: sleeping 0.04s (req=41, tokens=1976731)
⚠️ Throttling: sleeping 0.04s (req=41, tokens=1976747)
⚠️ Throttling: sleeping 0.01s (req=41, tokens=1976301)
⚠️ Throttling: sleeping 0.04s (req=41, tokens=1976266)
⚠️ Throttli

2025/11/10 23:59:32 INFO dspy.evaluate.evaluate: Average Metric: 8.0 / 144 (5.6%)


,paper_id,problem_root_rel,annotated_file_path,snippet_name,start_line,end_line,masked_file,context_files,test_file_path,paper,instance_id,result,rcb_score
0,advantage-alignment,pset/advantage-alignment,train.py,aa_terms3,883,898,import copy import hydra import math import numpy as np import os ...,[],paper2code_test.py,"{'format': 'tex', 'content': '\n\\documentclass{article} \\usepack...",advantage-alignment::train.py::883:898,"To implement ""aa_terms3"" in the `aa_terms` method, we need to foll...",
1,advantage-alignment,pset/advantage-alignment,train.py,aa_terms2,884,895,import copy import hydra import math import numpy as np import os ...,[],paper2code_test.py,"{'format': 'tex', 'content': '\n\\documentclass{article} \\usepack...",advantage-alignment::train.py::884:895,"**To implement aa_terms2 in AdvantageAlignment.aa_terms, following...",
2,advantage-alignment,pset/advantage-alignment,train.py,aa_terms1,885,887,import copy import hydra import math import numpy as np import os ...,[],paper2code_test.py,"{'format': 'tex', 'content': '\n\\documentclass{article} \\usepack...",advantage-alignment::train.py::885:887,"The code block to implement ""aa_terms1"" inside the `aa_terms` func...",
3,advantage-alignment,pset/advantage-alignment,train.py,integrated_aa,987,992,import copy import hydra import math import numpy as np import os ...,[],paper2code_test.py,"{'format': 'tex', 'content': '\n\\documentclass{article} \\usepack...",advantage-alignment::train.py::987:992,"To implement the block marked ""integrated_aa"" in the actor_losses ...",
4,advantage-alignment,pset/advantage-alignment,train.py,proximal_surrogate,995,1000,import copy import hydra import math import numpy as np import os ...,[],paper2code_test.py,"{'format': 'tex', 'content': '\n\\documentclass{article} \\usepack...",advantage-alignment::train.py::995:1000,"```python # Block ""proximal_surrogate"" implementation: ratios = to...",
...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,SISS,pset/SISS,losses.py,double forward passes,86,92,import torch import random import numpy as np class DDPMDeletionLo...,[],paper2code_test.py,"{'format': 'tex', 'content': ""\n\\documentclass{article} \\usepack...",SISS::losses.py::86:92,"To implement the ""double forward passes"" block in the `double_forw...",
140,Tanh-Init,pset/Tanh-Init,model.py,proposed weight initialization,9,28,import torch import numpy as np class Proposed: def __init__(self)...,[],paper2code_test.py,"{'format': 'tex', 'content': ""\n\\documentclass{article} \\usepack...",Tanh-Init::model.py::9:28,# Proposed Weight Initialization for Tanh Neural Networks The pape...,
141,Tanh-Init,pset/Tanh-Init,model.py,identity_matrix,11,21,import torch import numpy as np class Proposed: def __init__(self)...,[],paper2code_test.py,"{'format': 'tex', 'content': ""\n\\documentclass{article} \\usepack...",Tanh-Init::model.py::11:21,"```python # Block ""identity_matrix"" # Initialize an identity matri...",
142,Tanh-Init,pset/Tanh-Init,model.py,identity_matrix_else,15,20,import torch import numpy as np class Proposed: def __init__(self)...,[],paper2code_test.py,"{'format': 'tex', 'content': ""\n\\documentclass{article} \\usepack...",Tanh-Init::model.py::15:20,"```python\n# For m < n, construct an n x n identity, then take the...",


5.56

## Load the GEPA Optimizer

In [ ]:

import dspy
from gepa_artifact.gepa.gepa import GEPA,GEPAState
from gepa_artifact.utils.capture_stream_logger import Logger

import time

In [ ]:


runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if rc_metas[0].feedback_fn_maps is None or rc_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = rc_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = rc_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=rc_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    num_iters=20,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=9)

## Optimize the program with GEPA

In [ ]:
rc_metas[0].program[0].get_lm()

In [ ]:
# # load if any 
# state = GEPAState.load('runs/2025-11-11_00-25-53/')
# optimizer.gepa_state = state


In [ ]:
optimized_program = optimizer.compile(
    rc_metas[0].program[0],
    trainset=bench.train_set,
    valset=bench.val_set,
)

2025/11/11 00:57:08 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.


Average Metric: 0.00 / 2 (0.0%):  67%|██████▋   | 2/3 [01:04<00:32, 32.43s/it]


KeyboardInterrupt: 

In [ ]:
optimizer.gepa_state.save(runs_dir)



In [ ]:
def idxmax(lst):
    """Return the index of the maximum value in a list."""
    max_val = max(lst)
    return lst.index(max_val)

In [ ]:
gepa_state = optimizer.gepa_state
best_prog_idx = idxmax(gepa_state.per_program_tracked_scores)
best_prog = gepa_state.program_candidates[best_prog_idx]

In [ ]:
optimized_program=best_prog

## Now, let's evaluate the optimized program

In [ ]:
evaluate(optimized_program)

GEPA was able to optimize the base program **from 57% score to 61% score** in just 9 iterations. With higher budget, the optimized program's score can go as high as **64%**.

### Let's print the prompts that GEPA discovered

In [ ]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")